In [3]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 1: Setup
# Purpose: Initialize paths, device, imports for ablation study
# ============================================================

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from tqdm.auto import tqdm

PROJECT_ROOT = "/mnt/g/banglafake-detection"
MODELS_DIR = os.path.join(PROJECT_ROOT, "models")
REPORTS_DIR = os.path.join(PROJECT_ROOT, "reports")
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 256

print("Device:", device)

Device: cuda


In [4]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 2: Define Model Architectures (Baseline + Ablations + Full)
# ============================================================

class BanglaBERTOnly(nn.Module):
    def __init__(self, model_name, num_classes=2, dropout=0.1):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = outputs.last_hidden_state[:, 0, :]  # CLS token (ELECTRA has no pooler)
        output = self.dropout(cls_output)
        return self.classifier(output)


class BanglaBERTWithCNN(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(cnn_channels, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        pooled = torch.mean(x, dim=2)
        return self.classifier(self.dropout(pooled))


class BanglaBERTWithAttention(nn.Module):
    def __init__(self, model_name, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.attention_projection = nn.Linear(self.bert.config.hidden_size, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(self.dropout(attended))


class BanglaBERTFullModel(nn.Module):
    def __init__(self, model_name, cnn_channels=256, kernel_size=3, attention_dim=128, num_classes=2, dropout=0.3):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.cnn = nn.Conv1d(
            self.bert.config.hidden_size, cnn_channels, kernel_size,
            padding=kernel_size // 2
        )
        self.attention_projection = nn.Linear(cnn_channels, attention_dim)
        self.attention_score = nn.Linear(attention_dim, 1)
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(cnn_channels, num_classes)
        )

    def forward(self, input_ids, attention_mask):
        x = self.bert(input_ids=input_ids, attention_mask=attention_mask).last_hidden_state
        mask = attention_mask.unsqueeze(-1).float()
        x = x * mask
        x = x.permute(0, 2, 1)
        x = torch.relu(self.cnn(x))
        x = x.permute(0, 2, 1)
        att_hidden = torch.tanh(self.attention_projection(x))
        att_logits = self.attention_score(att_hidden).squeeze(-1)
        att_logits = att_logits.masked_fill(attention_mask == 0, -1e9)
        att_weights = torch.softmax(att_logits, dim=1)
        attended = (x * att_weights.unsqueeze(-1)).sum(dim=1)
        return self.classifier(attended)


print("Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel")

Architectures defined: BanglaBERTOnly, BanglaBERTWithCNN, BanglaBERTWithAttention, BanglaBERTFullModel


In [5]:
# ============================================================
# Cell 3: Load Data, Tokenizer, Dataset, DataLoaders
# ============================================================

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "validation.csv"))
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class BanglaFakeNewsDataset(Dataset):
    def __init__(self, dataframe, tokenizer, max_length=256):
        self.dataframe = dataframe
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        text = str(self.dataframe.iloc[idx]["text"])
        label = int(self.dataframe.iloc[idx]["label"])
        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }


train_loader = DataLoader(
    BanglaFakeNewsDataset(train_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=True, pin_memory=True
)
val_loader = DataLoader(
    BanglaFakeNewsDataset(val_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)
test_loader = DataLoader(
    BanglaFakeNewsDataset(test_df, tokenizer, MAX_LENGTH),
    batch_size=8, shuffle=False, pin_memory=True
)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))

Train: 2790 Val: 598 Test: 599


In [6]:
# ============================================================
# Cell 4: Train/Eval Helper Functions
# ============================================================

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_fn = nn.CrossEntropyLoss()
    total_loss = 0
    for batch in tqdm(loader, desc="Train", leave=False):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    return total_loss / len(loader)


def eval_split(model, loader, device):
    model.eval()
    preds, labels_all, probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Eval", leave=False):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            prob = torch.softmax(logits, dim=1)[:, 1]
            pred = torch.argmax(logits, dim=1)

            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())
            probs.extend(prob.cpu().numpy())
    return {
        "Accuracy": accuracy_score(labels_all, preds),
        "Precision": precision_score(labels_all, preds),
        "Recall": recall_score(labels_all, preds),
        "F1": f1_score(labels_all, preds),
        "ROC-AUC": roc_auc_score(labels_all, probs)
    }


print("Train/eval functions ready.")

Train/eval functions ready.


In [7]:
# ============================================================
# Cell 5: Train Ablation Models (BanglaBERT-only, +CNN, +Attention)
# Full train set (2790), same NUM_EPOCHS=3, best-val-F1 checkpoint
# ============================================================

NUM_EPOCHS = 3
ablation_results = {}

model_configs = {
    "BanglaBERT-only": lambda: BanglaBERTOnly(MODEL_NAME),
    "BanglaBERT+CNN": lambda: BanglaBERTWithCNN(MODEL_NAME),
    "BanglaBERT+Attention": lambda: BanglaBERTWithAttention(MODEL_NAME),
}

for name, ctor in model_configs.items():
    print(f"\n=== Training {name} ===")
    model = ctor().to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    best_val_f1, best_state = -1, None

    for epoch in range(NUM_EPOCHS):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_metrics = eval_split(model, val_loader, device)
        print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_f1={val_metrics['F1']:.4f}")
        if val_metrics['F1'] > best_val_f1:
            best_val_f1 = val_metrics['F1']
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(best_state, os.path.join(MODELS_DIR, f"ablation_{name.replace('+','_').replace(' ','_')}.pt"))
    test_metrics = eval_split(model, test_loader, device)
    ablation_results[name] = test_metrics
    print(f"{name} TEST:", test_metrics)

    del model
    torch.cuda.empty_cache()

print("\nAblation training complete.")


=== Training BanglaBERT-only ===


Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 1: train_loss=0.1621 val_f1=0.9771


Train:   0%|          | 0/349 [00:00<?, ?it/s]

...

Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Epoch 3: train_loss=0.0421 val_f1=0.9884


Eval:   0%|          | 0/75 [00:00<?, ?it/s]

BanglaBERT+Attention TEST: {'Accuracy': 0.9816360601001669, 'Precision': 0.9737704918032787, 'Recall': 0.99, 'F1': 0.9818181818181818, 'ROC-AUC': 0.995897435897436}

Ablation training complete.


In [8]:
# ============================================================
# Cell 6: Load Existing Full Model Checkpoint & Evaluate (sanity check)
# ============================================================

full_model = BanglaBERTFullModel(MODEL_NAME).to(device)
checkpoint_path = os.path.join(MODELS_DIR, "best_banglabert_cnn_attention.pt")
checkpoint = torch.load(checkpoint_path, map_location=device)
full_model.load_state_dict(checkpoint["model_state_dict"])

full_model_metrics = eval_split(full_model, test_loader, device)
ablation_results["BanglaBERT+CNN+Attention (Full)"] = full_model_metrics
print("Full Model TEST (should match Notebook 2 results ~0.9733 acc):", full_model_metrics)

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Full Model TEST (should match Notebook 2 results ~0.9733 acc): {'Accuracy': 0.9732888146911519, 'Precision': 0.9797297297297297, 'Recall': 0.9666666666666667, 'F1': 0.9731543624161074, 'ROC-AUC': 0.9926086956521739}


In [9]:
# ============================================================
# Cell 7: Final Ablation Comparison Table
# ============================================================

comparison_df = pd.DataFrame(ablation_results).T
comparison_df = comparison_df.round(4)
display(comparison_df)

output_path = os.path.join(REPORTS_DIR, "ablation_comparison_table.csv")
comparison_df.to_csv(output_path)
print("\nSaved:", output_path)

,Accuracy,Precision,Recall,F1,ROC-AUC
BanglaBERT-only,0.9800,0.9645,0.9967,0.9803,0.9944
BanglaBERT+CNN,0.9766,0.9735,0.9800,0.9767,0.9907
BanglaBERT+Attention,0.9816,0.9738,0.9900,0.9818,0.9959
BanglaBERT+CNN+Attention (Full),0.9733,0.9797,0.9667,0.9732,0.9926



Saved: /mnt/g/banglafake-detection/reports/ablation_comparison_table.csv


In [10]:
# ============================================================
# Cell 8: Collect Test Predictions for All Ablation Models
# ============================================================

all_preds = {}
true_labels = None

# Ablation models
ablation_paths = {
    "BanglaBERT-only": "ablation_BanglaBERT-only.pt",
    "BanglaBERT+CNN": "ablation_BanglaBERT_CNN.pt",
    "BanglaBERT+Attention": "ablation_BanglaBERT_Attention.pt",
}

ablation_classes = {
    "BanglaBERT-only": BanglaBERTOnly,
    "BanglaBERT+CNN": BanglaBERTWithCNN,
    "BanglaBERT+Attention": BanglaBERTWithAttention,
}

for name, fname in ablation_paths.items():
    model = ablation_classes[name](MODEL_NAME).to(device)
    model.load_state_dict(torch.load(os.path.join(MODELS_DIR, fname), map_location=device))
    model.eval()

    preds, labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            logits = model(input_ids, attention_mask)
            pred = torch.argmax(logits, dim=1)
            preds.extend(pred.cpu().numpy())
            labels.extend(batch['labels'].numpy())

    all_preds[name] = np.array(preds)
    if true_labels is None:
        true_labels = np.array(labels)

    del model
    torch.cuda.empty_cache()

# Full model
full_model = BanglaBERTFullModel(MODEL_NAME).to(device)
checkpoint = torch.load(
    os.path.join(MODELS_DIR, "best_banglabert_cnn_attention.pt"),
    map_location=device
)
full_model.load_state_dict(checkpoint["model_state_dict"])
full_model.eval()

preds = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        logits = full_model(input_ids, attention_mask)
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.cpu().numpy())

all_preds["BanglaBERT+CNN+Attention (Full)"] = np.array(preds)

del full_model
torch.cuda.empty_cache()

print("Predictions collected for:", list(all_preds.keys()))
print("true_labels shape:", true_labels.shape)

Predictions collected for: ['BanglaBERT-only', 'BanglaBERT+CNN', 'BanglaBERT+Attention', 'BanglaBERT+CNN+Attention (Full)']
true_labels shape: (599,)


In [11]:
# ============================================================
# Cell 9: McNemar's Test — Baseline vs Each Variant
# Purpose: Check if performance differences are statistically significant
# ============================================================

from statsmodels.stats.contingency_tables import mcnemar

correct = {name: (preds == true_labels).astype(int) for name, preds in all_preds.items()}

baseline_name = "BanglaBERT-only"
mcnemar_results = []

for name in all_preds:
    if name == baseline_name:
        continue
    b_correct = correct[baseline_name]
    v_correct = correct[name]

    n01 = np.sum((b_correct == 1) & (v_correct == 0))
    n10 = np.sum((b_correct == 0) & (v_correct == 1))
    n00 = np.sum((b_correct == 0) & (v_correct == 0))
    n11 = np.sum((b_correct == 1) & (v_correct == 1))

    table = [[n11, n01], [n10, n00]]
    result = mcnemar(table, exact=(n01 + n10 < 25), correction=True)

    mcnemar_results.append({
        "Comparison": f"{baseline_name} vs {name}",
        "Baseline_correct_only": n01,
        "Variant_correct_only": n10,
        "statistic": result.statistic,
        "p_value": result.pvalue,
        "Significant (p<0.05)": result.pvalue < 0.05
    })

mcnemar_df = pd.DataFrame(mcnemar_results)
display(mcnemar_df)

mcnemar_df.to_csv(os.path.join(REPORTS_DIR, "mcnemar_test_results.csv"), index=False)
print("\nSaved:", os.path.join(REPORTS_DIR, "mcnemar_test_results.csv"))

,Comparison,Baseline_correct_only,Variant_correct_only,statistic,p_value,Significant (p<0.05)
0,BanglaBERT-only vs BanglaBERT+CNN,7,5,5.0,0.774414,False
1,BanglaBERT-only vs BanglaBERT+Attention,3,4,3.0,1.000000,False
2,BanglaBERT-only vs BanglaBERT+CNN+Attention (F...,9,5,5.0,0.423950,False



Saved: /mnt/g/banglafake-detection/reports/mcnemar_test_results.csv


In [12]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 10: Multi-Seed Robustness Check (3 seeds) — All 4 Models
# Purpose: Verify if Baseline > Full model ranking holds across seeds
# ============================================================

import random

SEEDS = [42, 123, 2024]
NUM_EPOCHS = 3
multi_seed_results = []

model_configs = {
    "BanglaBERT-only": lambda: BanglaBERTOnly(MODEL_NAME),
    "BanglaBERT+CNN": lambda: BanglaBERTWithCNN(MODEL_NAME),
    "BanglaBERT+Attention": lambda: BanglaBERTWithAttention(MODEL_NAME),
    "BanglaBERT+CNN+Attention (Full)": lambda: BanglaBERTFullModel(MODEL_NAME),
}

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

for seed in SEEDS:
    print(f"\n{'='*20} SEED {seed} {'='*20}")
    set_seed(seed)

    # Rebuild loaders with seeded shuffling
    g = torch.Generator()
    g.manual_seed(seed)
    train_loader_seed = DataLoader(BanglaFakeNewsDataset(train_df, tokenizer, MAX_LENGTH),
                                    batch_size=8, shuffle=True, generator=g, pin_memory=True)

    for name, ctor in model_configs.items():
        print(f"\n--- {name} (seed {seed}) ---")
        model = ctor().to(device)
        optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        best_val_f1, best_state = -1, None

        for epoch in range(NUM_EPOCHS):
            train_loss = train_one_epoch(model, train_loader_seed, optimizer, device)
            val_metrics = eval_split(model, val_loader, device)
            if val_metrics['F1'] > best_val_f1:
                best_val_f1 = val_metrics['F1']
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        model.load_state_dict(best_state)
        test_metrics = eval_split(model, test_loader, device)
        test_metrics["Model"] = name
        test_metrics["Seed"] = seed
        multi_seed_results.append(test_metrics)
        print(f"{name} seed={seed} TEST:", test_metrics)

        del model, optimizer
        torch.cuda.empty_cache()

multi_seed_df = pd.DataFrame(multi_seed_results)
summary_df = multi_seed_df.groupby("Model")[["Accuracy","Precision","Recall","F1","ROC-AUC"]].agg(['mean','std']).round(4)

display(multi_seed_df)
display(summary_df)

multi_seed_df.to_csv(os.path.join(REPORTS_DIR, "multi_seed_raw_results.csv"), index=False)
summary_df.to_csv(os.path.join(REPORTS_DIR, "multi_seed_summary.csv"))
print("\nSaved multi_seed_raw_results.csv and multi_seed_summary.csv")


==================== SEED 42 ====================

--- BanglaBERT-only (seed 42) ---


Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

Train:   0%|          | 0/349 [00:00<?, ?it/s]

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

...

Eval:   0%|          | 0/75 [00:00<?, ?it/s]

BanglaBERT+CNN+Attention (Full) seed=2024 TEST: {'Accuracy': 0.9799666110183639, 'Precision': 0.98, 'Recall': 0.98, 'F1': 0.98, 'ROC-AUC': 0.994314381270903, 'Model': 'BanglaBERT+CNN+Attention (Full)', 'Seed': 2024}


,Accuracy,Precision,Recall,F1,ROC-AUC,Model,Seed
0,0.979967,0.980000,0.980000,0.980000,0.992051,BanglaBERT-only,42
1,0.979967,0.964516,0.996667,0.980328,0.991594,BanglaBERT+CNN,42
2,0.974958,0.961165,0.990000,0.975369,0.993478,BanglaBERT+Attention,42
3,0.979967,0.973684,0.986667,0.980132,0.996767,BanglaBERT+CNN+Attention (Full),42
4,0.973289,0.955128,0.993333,0.973856,0.993311,BanglaBERT-only,123
5,0.983306,0.970779,0.996667,0.983553,0.992832,BanglaBERT+CNN,123
6,0.986644,0.980263,0.993333,0.986755,0.996076,BanglaBERT+Attention,123
7,0.971619,0.979661,0.963333,0.971429,0.994103,BanglaBERT+CNN+Attention (Full),123
8,0.979967,0.980000,0.980000,0.980000,0.992564,BanglaBERT-only,2024
9,0.973289,0.982993,0.963333,0.973064,0.983266,BanglaBERT+CNN,2024


Accuracy         Precision          Recall  \
                                    mean     std      mean     std    mean   
Model                                                                        
BanglaBERT+Attention              0.9783  0.0073    0.9737  0.0109  0.9833   
BanglaBERT+CNN                    0.9789  0.0051    0.9728  0.0094  0.9856   
BanglaBERT+CNN+Attention (Full)   0.9772  0.0048    0.9778  0.0036  0.9767   
BanglaBERT-only                   0.9777  0.0039    0.9717  0.0144  0.9844   

                                             F1         ROC-AUC          
                                    std    mean     std    mean     std  
Model                                                                    
BanglaBERT+Attention             0.0145  0.9784  0.0073  0.9946  0.0013  
BanglaBERT+CNN                   0.0192  0.9790  0.0054  0.9892  0.0052  
BanglaBERT+CNN+Attention (Full)  0.0120  0.9772  0.0050  0.9951  0.0015  
BanglaBERT-only                  0.0077  0.9780  0.0035  0.9926  0.0006


Saved multi_seed_raw_results.csv and multi_seed_summary.csv


In [13]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 11: Repeated-Measures ANOVA — Confirm No Significant 
#          Architecture Effect Across Seeds
# ============================================================

from statsmodels.stats.anova import AnovaRM

# Reshape for repeated-measures ANOVA (Model = within-subject factor, Seed = subject)
anova_df = multi_seed_df[["Model", "Seed", "Accuracy", "F1"]].copy()

print("=== Repeated-Measures ANOVA: Accuracy ===")
aovrm_acc = AnovaRM(anova_df, depvar='Accuracy', subject='Seed', within=['Model'])
res_acc = aovrm_acc.fit()
print(res_acc.summary())

print("\n=== Repeated-Measures ANOVA: F1 ===")
aovrm_f1 = AnovaRM(anova_df, depvar='F1', subject='Seed', within=['Model'])
res_f1 = aovrm_f1.fit()
print(res_f1.summary())

# Save results
anova_results = pd.DataFrame({
    "Metric": ["Accuracy", "F1"],
    "F_statistic": [res_acc.anova_table["F Value"].values[0], res_f1.anova_table["F Value"].values[0]],
    "p_value": [res_acc.anova_table["Pr > F"].values[0], res_f1.anova_table["Pr > F"].values[0]]
})
anova_results["Significant (p<0.05)"] = anova_results["p_value"] < 0.05

display(anova_results)
anova_results.to_csv(os.path.join(REPORTS_DIR, "anova_architecture_effect.csv"), index=False)
print("\nSaved: anova_architecture_effect.csv")

=== Repeated-Measures ANOVA: Accuracy ===
              Anova
      F Value Num DF Den DF Pr > F
----------------------------------
Model  0.0418 3.0000 6.0000 0.9875


=== Repeated-Measures ANOVA: F1 ===
              Anova
      F Value Num DF Den DF Pr > F
----------------------------------
Model  0.0464 3.0000 6.0000 0.9855



,Metric,F_statistic,p_value,Significant (p<0.05)
0,Accuracy,0.041754,0.987517,False
1,F1,0.046410,0.985462,False



Saved: anova_architecture_effect.csv


In [14]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 12: Inspect Source Distribution for Group-Based Split
# Purpose: Verify source column across train/val/test before
#          building a source-held-out generalization experiment
# ============================================================

# Load raw processed data (need Source column - check if present)
train_full = pd.read_csv(os.path.join(PROCESSED_DIR, "train.csv"))
val_full = pd.read_csv(os.path.join(PROCESSED_DIR, "validation.csv"))
test_full = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv"))

print("Train columns:", train_full.columns.tolist())
print("\nDoes 'Source' column exist in train.csv?", "Source" in train_full.columns)

if "Source" in train_full.columns:
    all_data = pd.concat([train_full, val_full, test_full])
    source_counts = all_data["Source"].value_counts()
    print("\nTotal unique sources:", source_counts.shape[0])
    print("\nTop 15 sources:")
    print(source_counts.head(15))

    source_label = all_data.groupby("Source")["label"].nunique()
    mixed_sources = source_label[source_label > 1].index.tolist()
    print("\nMixed-label sources:", mixed_sources)
else:
    print("\nWARNING: 'Source' column not in processed CSVs. Need to check Notebook 1 output or raw data for source info.")

Train columns: ['Global_ID', 'text', 'label']

Does 'Source' column exist in train.csv? False



In [15]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 13: Reconstruct Source Metadata via Raw Data Merge
# Purpose: Recover 'Source' column for train/val/test by 
#          merging on reconstructed 'text' (same method as Notebook 3)
# ============================================================

real_raw = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "Bangla_Real_News_Dataset.csv"))
fake_raw = pd.read_csv(os.path.join(PROJECT_ROOT, "data", "raw", "Bangla_Fake_News_Dataset.csv"))
raw_df = pd.concat([real_raw, fake_raw], ignore_index=True)

import re
def clean_text_for_analysis(text):
    if pd.isna(text):
        return ""
    text = str(text).strip()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

raw_df["text"] = (raw_df["Headline"].apply(clean_text_for_analysis) + " " +
                   raw_df["Content"].apply(clean_text_for_analysis)).str.strip()

source_map = raw_df[["text", "Source"]].drop_duplicates(subset=["text"])

train_src = train_df.merge(source_map, on="text", how="left")
val_src   = val_df.merge(source_map, on="text", how="left")
test_src  = test_df.merge(source_map, on="text", how="left")

print("Train missing Source:", train_src["Source"].isna().sum(), "/", len(train_src))
print("Val missing Source:", val_src["Source"].isna().sum(), "/", len(val_src))
print("Test missing Source:", test_src["Source"].isna().sum(), "/", len(test_src))

all_src = pd.concat([train_src, val_src, test_src])
print("\nUnique sources total:", all_src["Source"].nunique())

source_label_counts = all_src.groupby("Source")["label"].nunique()
mixed_sources = source_label_counts[source_label_counts > 1].index.tolist()
print("Mixed-label sources:", mixed_sources)

print("\nTop 10 sources by count:")
print(all_src["Source"].value_counts().head(10))

Train missing Source: 0 / 2790
Val missing Source: 0 / 598
Test missing Source: 0 / 599

Unique sources total: 45
Mixed-label sources: ['Reporter', 'Reuters']

Top 10 sources by count:
Source
Reporter            2443
rumorscanner.com    1116
fact-watch.org       301
Reuters               26
BBC                   10
jachai.org            10
Al Jazeera             8
AFP                    7
Bold Sky               6
CNN                    6
Name: count, dtype: int64


In [16]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 14: Build Source-Held-Out (Group) Split
# Purpose: Ensure no source appears in both train and test —
#          tests true generalization vs source-shortcut learning
# ============================================================

from sklearn.model_selection import GroupShuffleSplit

full_df = pd.concat([train_src, val_src, test_src], ignore_index=True)
print("Full dataset shape:", full_df.shape)
print("Label distribution (full):", full_df["label"].value_counts().to_dict())

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, test_idx = next(gss.split(full_df, groups=full_df["Source"]))

group_train_df = full_df.iloc[train_idx].reset_index(drop=True)
group_test_df  = full_df.iloc[test_idx].reset_index(drop=True)

# Further split group_train_df into train/val (source-aware, 90/10)
gss_val = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=42)
tr_idx, val_idx = next(gss_val.split(group_train_df, groups=group_train_df["Source"]))
group_train_final = group_train_df.iloc[tr_idx].reset_index(drop=True)
group_val_final   = group_train_df.iloc[val_idx].reset_index(drop=True)

print("\n=== Source-Held-Out Split ===")
print("Train:", group_train_final.shape, group_train_final["label"].value_counts().to_dict())
print("Val:", group_val_final.shape, group_val_final["label"].value_counts().to_dict())
print("Test:", group_test_df.shape, group_test_df["label"].value_counts().to_dict())

train_sources = set(group_train_final["Source"].unique()) | set(group_val_final["Source"].unique())
test_sources = set(group_test_df["Source"].unique())
print("\nSource overlap between (train+val) and test:", train_sources & test_sources)
print("Test-only sources:", test_sources)

Full dataset shape: (3987, 4)
Label distribution (full): {1: 1997, 0: 1990}

=== Source-Held-Out Split ===
Train: (3656, 4) {1: 1980, 0: 1676}
Val: (314, 4) {0: 301, 1: 13}
Test: (17, 4) {0: 13, 1: 4}

Source overlap between (train+val) and test: set()
Test-only sources: {'dhakafactcheck.com', 'The Tribune', 'Arab News', 'jachai.org', 'Science Times', 'bangla.aajtak', 'Reuters '}


In [17]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 15: Stratified Group Split (Balance-Aware, Source-Held-Out)
# Purpose: Better source-held-out split that also tries to 
#          preserve class balance across folds
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

fold_summaries = []
for fold, (tr_idx, te_idx) in enumerate(sgkf.split(full_df, y=full_df["label"], groups=full_df["Source"])):
    te_fold = full_df.iloc[te_idx]
    fold_summaries.append({
        "Fold": fold,
        "Test_size": len(te_fold),
        "Test_Fake": (te_fold["label"] == 0).sum(),
        "Test_Real": (te_fold["label"] == 1).sum(),
        "Test_sources": te_fold["Source"].nunique()
    })

fold_df = pd.DataFrame(fold_summaries)
display(fold_df)
print("\nPick the most balanced fold (closest to 50/50 Fake/Real) with reasonable size for the held-out test.")

,Fold,Test_size,Test_Fake,Test_Real,Test_sources
0,0,2443,554,1889,1
1,1,1123,1116,7,2
2,2,315,301,14,3
3,3,53,9,44,19
4,4,53,10,43,20



Pick the most balanced fold (closest to 50/50 Fake/Real) with reasonable size for the held-out test.


In [18]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 16: Source-Held-Out Experiment (Fold 4) — BanglaBERT-only
# Purpose: Train on 19/20 held-in sources, test on genuinely 
#          unseen sources; report balanced accuracy given imbalance
# ============================================================

from sklearn.metrics import balanced_accuracy_score

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(sgkf.split(full_df, y=full_df["label"], groups=full_df["Source"]))
tr_idx, te_idx = splits[4]  # Fold 4

group_train_df = full_df.iloc[tr_idx].reset_index(drop=True)
group_test_df  = full_df.iloc[te_idx].reset_index(drop=True)

print("Held-in train size:", len(group_train_df), group_train_df["label"].value_counts().to_dict())
print("Held-out test size:", len(group_test_df), group_test_df["label"].value_counts().to_dict())
print("Held-out sources:", group_test_df["Source"].unique().tolist())

# 90/10 stratified-group split of held-in data for train/val
sgkf_val = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
tr2_idx, val2_idx = next(sgkf_val.split(group_train_df, y=group_train_df["label"], groups=group_train_df["Source"]))
group_train_final = group_train_df.iloc[tr2_idx].reset_index(drop=True)
group_val_final   = group_train_df.iloc[val2_idx].reset_index(drop=True)

print("\nFinal train:", len(group_train_final), "Final val:", len(group_val_final))

group_train_loader = DataLoader(BanglaFakeNewsDataset(group_train_final, tokenizer, MAX_LENGTH),
                                 batch_size=8, shuffle=True, pin_memory=True)
group_val_loader = DataLoader(BanglaFakeNewsDataset(group_val_final, tokenizer, MAX_LENGTH),
                               batch_size=8, shuffle=False, pin_memory=True)
group_test_loader = DataLoader(BanglaFakeNewsDataset(group_test_df, tokenizer, MAX_LENGTH),
                                batch_size=8, shuffle=False, pin_memory=True)

set_seed(42)
model = BanglaBERTOnly(MODEL_NAME).to(device)
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
best_val_f1, best_state = -1, None

for epoch in range(NUM_EPOCHS):
    train_loss = train_one_epoch(model, group_train_loader, optimizer, device)
    val_metrics = eval_split(model, group_val_loader, device)
    print(f"Epoch {epoch+1}: train_loss={train_loss:.4f} val_f1={val_metrics['F1']:.4f}")
    if val_metrics['F1'] > best_val_f1:
        best_val_f1 = val_metrics['F1']
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)

# Evaluate on held-out (unseen) sources
model.eval()
preds, labels_all, probs = [], [], []
with torch.no_grad():
    for batch in group_test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        logits = model(input_ids, attention_mask)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = torch.argmax(logits, dim=1)
        preds.extend(pred.cpu().numpy())
        labels_all.extend(labels.cpu().numpy())
        probs.extend(prob.cpu().numpy())

acc = accuracy_score(labels_all, preds)
bal_acc = balanced_accuracy_score(labels_all, preds)
f1 = f1_score(labels_all, preds)

print("\n=== Source-Held-Out Test Results (unseen sources) ===")
print("Accuracy:", acc)
print("Balanced Accuracy:", bal_acc)
print("F1:", f1)
print("Predictions:", preds)
print("True labels:", labels_all)

torch.save(best_state, os.path.join(MODELS_DIR, "source_heldout_banglabert_only.pt"))

source_heldout_result = pd.DataFrame([{
    "Experiment": "Source-Held-Out (Fold 4, unseen sources)",
    "Test_size": len(group_test_df),
    "Accuracy": acc,
    "Balanced_Accuracy": bal_acc,
    "F1": f1
}])
source_heldout_result.to_csv(os.path.join(REPORTS_DIR, "source_heldout_results.csv"), index=False)
print("\nSaved: source_heldout_results.csv")

Held-in train size: 3934 {0: 1980, 1: 1954}
Held-out test size: 53 {1: 43, 0: 10}
Held-out sources: ['BBC', 'Ei Somoy', 'Bold Sky', 'Arab News', 'The Guardian', 'jachai.org', 'Health Line', 'Daily Sun', 'ND TV', 'Reuters ', 'New York Times', 'Reuters\n', 'Amazon', 'UNB', 'National Geography', 'France24', 'India Today', 'Science Times', 'The Times', 'The Business Standard']

Final train: 2818 Final val: 1116


Train:   0%|          | 0/353 [00:00<?, ?it/s]

Eval:   0%|          | 0/140 [00:00<?, ?it/s]

/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 1: train_loss=0.1571 val_f1=0.0000


...

/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Train:   0%|          | 0/353 [00:00<?, ?it/s]

Eval:   0%|          | 0/140 [00:00<?, ?it/s]

/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_classification.py:1879: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/jaimul/pytorch-env/lib/python3.12/site-packages/sklearn/metrics/_ranking.py:469: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(


Epoch 3: train_loss=0.0318 val_f1=0.0000

=== Source-Held-Out Test Results (unseen sources) ===
Accuracy: 0.9245283018867925
Balanced Accuracy: 0.8383720930232558
F1: 0.9545454545454546
Predictions: [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1)]
True labels: [np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int

In [19]:
# ============================================================
# Notebook: 05_ablation_interpretability.ipynb
# Cell 17: Bootstrap Confidence Interval for Source-Held-Out Result
# Purpose: Quantify uncertainty around balanced accuracy (n=53)
# ============================================================

n_bootstrap = 2000
rng = np.random.RandomState(42)

preds_arr = np.array(preds)
labels_arr = np.array(labels_all)
n = len(labels_arr)

boot_bal_acc = []
boot_acc = []

for _ in range(n_bootstrap):
    idx = rng.choice(n, size=n, replace=True)
    if len(np.unique(labels_arr[idx])) < 2:
        continue
    boot_bal_acc.append(balanced_accuracy_score(labels_arr[idx], preds_arr[idx]))
    boot_acc.append(accuracy_score(labels_arr[idx], preds_arr[idx]))

boot_bal_acc = np.array(boot_bal_acc)
boot_acc = np.array(boot_acc)

ci_bal_acc = np.percentile(boot_bal_acc, [2.5, 97.5])
ci_acc = np.percentile(boot_acc, [2.5, 97.5])

print("Balanced Accuracy: {:.4f}  95% CI: [{:.4f}, {:.4f}]".format(bal_acc, *ci_bal_acc))
print("Accuracy: {:.4f}  95% CI: [{:.4f}, {:.4f}]".format(acc, *ci_acc))
print("Valid bootstrap resamples used:", len(boot_bal_acc), "/", n_bootstrap)

ci_results = pd.DataFrame([{
    "Metric": "Balanced Accuracy",
    "Point_Estimate": bal_acc,
    "CI_Lower": ci_bal_acc[0],
    "CI_Upper": ci_bal_acc[1],
    "N": n
}, {
    "Metric": "Accuracy",
    "Point_Estimate": acc,
    "CI_Lower": ci_acc[0],
    "CI_Upper": ci_acc[1],
    "N": n
}])
display(ci_results)
ci_results.to_csv(os.path.join(REPORTS_DIR, "source_heldout_ci.csv"), index=False)
print("\nSaved: source_heldout_ci.csv")

Balanced Accuracy: 0.8384  95% CI: [0.6764, 0.9796]
Accuracy: 0.9245  95% CI: [0.8491, 0.9811]
Valid bootstrap resamples used: 2000 / 2000


,Metric,Point_Estimate,CI_Lower,CI_Upper,N
0,Balanced Accuracy,0.838372,0.676389,0.979592,53
1,Accuracy,0.924528,0.849057,0.981132,53



Saved: source_heldout_ci.csv
